In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install mne tensorflow scipy scikit-learn pandas tqdm -q

In [3]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import mne
from scipy.signal import welch

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, Activation
from tensorflow.keras.layers import SpatialDropout1D, Add, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

2026-04-24 17:58:23.101531: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777053503.297395      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777053503.362493      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777053503.812636      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777053503.812680      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777053503.812683      55 computation_placer.cc:177] computation placer alr

In [4]:
import subprocess
import os

# Install AWS CLI if not available
subprocess.run(['pip', 'install', 'awscli', '-q'], check=True)

# Create the target directory
os.makedirs('/kaggle/working/ds004504/derivatives', exist_ok=True)

# Download just the derivatives folder from OpenNeuro's public S3 bucket
# No AWS account needed — this bucket is publicly accessible
result = subprocess.run([
    'aws', 's3', 'sync',
    's3://openneuro.org/ds004504/derivatives/',
    '/kaggle/working/ds004504/derivatives/',
    '--no-sign-request',
    '--region', 'us-east-1'
], capture_output=True, text=True)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode == 0:
    print("\nDownload complete.")
else:
    print("Error:", result.stderr[-1000:])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 33.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
aiobotocore 3.3.0 requires botocore<1.42.71,>=1.42.62, but you have botocore 1.42.95 which is incompatible.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


 remaining
Completed 2.7 GiB/2.7 GiB (181.0 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (181.0 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (181.0 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (181.0 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (181.0 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (181.0 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 GiB/2.7 GiB (180.9 MiB/s) with 2 file(s) remaining
Completed 2.7 

In [5]:
# Download the participants.tsv file which has the diagnostic labels.
result2 = subprocess.run([
    'aws', 's3', 'cp',
    's3://openneuro.org/ds004504/participants.tsv',
    '/kaggle/working/ds004504/participants.tsv',
    '--no-sign-request',
    '--region', 'us-east-1'
], capture_output=True, text=True)

print(result2.stdout)
if result2.returncode == 0:
    print("participants.tsv downloaded.")
else:
    print("Error:", result2.stderr)

Completed 1.7 KiB/1.7 KiB (5.6 KiB/s) with 1 file(s) remaining
download: s3://openneuro.org/ds004504/participants.tsv to ds004504/participants.tsv

participants.tsv downloaded.


In [6]:
DATA_PATH = Path("/kaggle/working/ds004504/")
EEG_PATH = DATA_PATH / "derivatives"
LABELS_PATH = DATA_PATH / "participants.tsv"

In [7]:
participants = pd.read_csv(LABELS_PATH, sep="\t")

print(participants.columns)

group_map = {
    'A': 0,   # AD
    'F': 1,   # FTD
    'C': 2    # Healthy
}

participants['label'] = participants['Group'].map(group_map)
participants.head()

Index(['participant_id', 'Gender', 'Age', 'Group', 'MMSE'], dtype='object')


,participant_id,Gender,Age,Group,MMSE,label
0,sub-001,F,57,A,16,0
1,sub-002,F,78,A,22,0
2,sub-003,M,70,A,14,0
3,sub-004,F,67,A,20,0
4,sub-005,M,70,A,22,0


In [8]:
eeg_files = []

for sub_folder in EEG_PATH.iterdir():
    if sub_folder.is_dir():
        eeg_folder = sub_folder / "eeg"
        
        if eeg_folder.exists():
            for f in eeg_folder.glob("*.set"):
                eeg_files.append({
                    "subject_id": sub_folder.name,
                    "filepath": str(f)
                })

eeg_df = pd.DataFrame(eeg_files)

eeg_df = eeg_df.merge(
    participants[['participant_id', 'label']],
    left_on='subject_id',
    right_on='participant_id',
    how='left'
)

print(eeg_df.head())
print("Total files:", len(eeg_df))

  subject_id                                           filepath  \
0    sub-031  /kaggle/working/ds004504/derivatives/sub-031/e...   
1    sub-071  /kaggle/working/ds004504/derivatives/sub-071/e...   
2    sub-081  /kaggle/working/ds004504/derivatives/sub-081/e...   
3    sub-003  /kaggle/working/ds004504/derivatives/sub-003/e...   
4    sub-069  /kaggle/working/ds004504/derivatives/sub-069/e...   

  participant_id  label  
0        sub-031      0  
1        sub-071      1  
2        sub-081      1  
3        sub-003      0  
4        sub-069      1  
Total files: 88


In [9]:
def create_epochs(raw):
    events = mne.make_fixed_length_events(raw, duration=6.0, overlap=3.0)
    epochs = mne.Epochs(raw, events, tmin=0, tmax=6.0, baseline=None, preload=True)
    return epochs

In [10]:
bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 16),
    "zaeta": (16, 24),
    "beta": (24, 30),
    "gamma": (30, 45)
}

In [11]:
def compute_rbp(epoch_data, sfreq):
    psd_all = []
    
    for ch in epoch_data:
        freqs, psd = welch(ch, fs=sfreq, nperseg=int(sfreq*2))
        total_power = np.sum(psd[(freqs >= 0.5) & (freqs <= 45)])
        
        band_powers = []
        for low, high in bands.values():
            band_power = np.sum(psd[(freqs >= low) & (freqs <= high)])
            rbp = band_power / total_power if total_power > 0 else 0
            band_powers.append(rbp)
        
        psd_all.append(band_powers)
    
    return np.mean(psd_all, axis=0)

In [12]:
X = []
y = []
subject_ids = []

for _, row in tqdm(eeg_df.iterrows(), total=len(eeg_df)):
    filepath = row['filepath']
    label = row['label']
    subject = row['subject_id']
    
    try:
        raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
        
        # DO NOT FILTER AGAIN
        
        epochs = create_epochs(raw)
        data = epochs.get_data()
        
        for epoch in data:
            features = compute_rbp(epoch, raw.info['sfreq'])
            
            X.append(features)
            y.append(label)
            subject_ids.append(subject)
            
    except Exception as e:
        print(f"Error: {filepath}, {e}")

X = np.array(X)
y = np.array(y)
subject_ids = np.array(subject_ids)

print("Final:", X.shape)

  0%|          | 0/88 [00:00<?, ?it/s]

Not setting metadata
382 matching events found


/tmp/ipykernel_55/2452335090.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 382 events and 3001 original time points ...
0 bad epochs dropped


  1%|          | 1/88 [00:09<14:28,  9.98s/it]

...

Not setting metadata
272 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 272 events and 3001 original time points ...
0 bad epochs dropped


/tmp/ipykernel_55/2452335090.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
 99%|█████████▉| 87/88 [06:59<00:05,  5.25s/it]

Not setting metadata
263 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 263 events and 3001 original time points ...
0 bad epochs dropped


/tmp/ipykernel_55/2452335090.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
100%|██████████| 88/88 [07:04<00:00,  4.82s/it]

Final: (23150, 6)


In [13]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

X = X.reshape(-1, 6, 1)

In [14]:
def build_model(num_classes):
    inp = Input(shape=(6,1))
    
    x = Conv1D(32, 7, padding='same')(inp)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = SpatialDropout1D(0.2)(x)
    
    res = Conv1D(32, 1, padding='same')(inp)
    
    x = Conv1D(32, 7, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Add()([x, res])
    
    x = LSTM(64)(x)
    
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.2)(x)
    
    x = Dense(192, activation='relu')(x)
    x = Dropout(0.2)(x)
    
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)
    
    out = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inp, out)
    
    model.compile(
        optimizer=Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [15]:
def run_task(name, X, y, subject_ids, keep_labels, merge_map):
    
    print(f"\n===== {name} =====")
    
    # Filter labels
    mask = np.isin(y, keep_labels)
    
    X_task = X[mask]
    y_task = y[mask]
    subj_task = subject_ids[mask]
    
    # Remap labels
    y_task = np.array([merge_map[val] for val in y_task])
    
    from sklearn.model_selection import GroupShuffleSplit
    
    # --- Train / Temp split (subject-wise) ---
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, temp_idx = next(gss.split(X_task, y_task, groups=subj_task))
    
    X_train, X_temp = X_task[train_idx], X_task[temp_idx]
    y_train, y_temp = y_task[train_idx], y_task[temp_idx]
    subj_temp = subj_task[temp_idx]
    
    # --- Val / Test split (subject-wise) ---
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    val_idx, test_idx = next(gss_val.split(X_temp, y_temp, groups=subj_temp))
    
    X_val, X_test = X_temp[val_idx], X_temp[test_idx]
    y_val, y_test = y_temp[val_idx], y_temp[test_idx]
    
    # --- Sanity check (NO LEAKAGE) ---
    train_sub = set(subj_task[train_idx])
    val_sub = set(subj_temp[val_idx])
    test_sub = set(subj_temp[test_idx])
    
    print("Overlap Train-Val:", train_sub & val_sub)
    print("Overlap Train-Test:", train_sub & test_sub)
    
    # --- One-hot encoding ---
    num_classes = len(np.unique(y_task))
    
    y_train = tf.keras.utils.to_categorical(y_train, num_classes)
    y_val = tf.keras.utils.to_categorical(y_val, num_classes)
    y_test = tf.keras.utils.to_categorical(y_test, num_classes)
    
    # --- Model ---
    model = build_model(num_classes)
    
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
        verbose=1
    )
    
    loss, acc = model.evaluate(X_test, y_test)
    print(f"{name} Test Accuracy:", acc)

In [16]:
# TASK 1 — AD vs Healthy
run_task(
    "AD vs Healthy",
    X, y, subject_ids,
    keep_labels=[0, 2],
    merge_map={0:0, 2:1}
)

# TASK 2 — FTD vs Healthy
run_task(
    "FTD vs Healthy",
    X, y, subject_ids,
    keep_labels=[1, 2],
    merge_map={1:0, 2:1}
)

# TASK 3 — AD+FTD vs Healthy
run_task(
    "AD+FTD vs Healthy",
    X, y, subject_ids,
    keep_labels=[0,1,2],
    merge_map={0:0, 1:0, 2:1}
)

# TASK 4 — AD vs FTD vs Healthy
run_task(
    "AD vs FTD vs Healthy",
    X, y, subject_ids,
    keep_labels=[0,1,2],
    merge_map={0:0, 1:1, 2:2}
)


===== AD vs Healthy =====
Overlap Train-Val: set()
Overlap Train-Test: set()


I0000 00:00:1777053983.653405      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/100


I0000 00:00:1777053988.998569     138 cuda_dnn.cc:529] Loaded cuDNN version 91002


433/433 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.5471 - loss: 0.6839 - val_accuracy: 0.4997 - val_loss: 0.6828
Epoch 2/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6596 - loss: 0.6221 - val_accuracy: 0.7607 - val_loss: 0.5441
Epoch 3/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6851 - loss: 0.5956 - val_accuracy: 0.7624 - val_loss: 0.5370
Epoch 4/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6925 - loss: 0.5893 - val_accuracy: 0.7607 - val_loss: 0.5440
Epoch 5/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6962 - loss: 0.5877 - val_accuracy: 0.7231 - val_loss: 0.5374
Epoch 6/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.7069 - loss: 0.5760 - val_accuracy: 0.7470 - val_loss: 0.5349
Epoch 7/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.7161 - loss: 0.5651 - val_accuracy: 0.7339 - val_loss: 0.5429
Epoch 8/100
433/433 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.7115 - loss: 0.5727 - val_accurac